# Build RAG from Scratch with qwen3-max

This notebook rewrites the Python script into Jupyter-style modular code.

# Package installation commands

Run the following commands in **Anaconda Prompt** or **PowerShell** before opening this notebook:
Create a conda env named **rag_qwen3max** 
```bash
conda create -n rag_qwen3max python=3.10 -y
conda activate rag_qwen3max
python -m pip install -U pip setuptools wheel
conda install -c conda-forge "numpy=1.26.4" "faiss-cpu=1.8.0" -y
pip install "langchain==0.3.27" "langchain-core==0.3.80" "langchain-community==0.3.31" "langchain-openai==0.3.35" "langchain-text-splitters==0.3.11" "tiktoken==0.8.0" "beautifulsoup4==4.12.3" "requests>=2.32.5,<3"
pip install ipykernel jupyter
python -m ipykernel install --user --name rag_qwen3max --display-name "Python (rag_qwen3max)"
pip check
```
Then select the python kernel **rag_qwen3max** in your notebook

---
This notebook follows the RAG flow:

**Indexing → Retrieval → Generation → Full RAG chain**

Blog 网页 
  ↓
WebBaseLoader 加载网页
  ↓
Document 原始文档
  ↓
RecursiveCharacterTextSplitter 切分
  ↓
Document chunks 文档块
  ↓
Embedding 模型 text-embedding-v4
  ↓
vectors 向量
  ↓
FAISS DB 向量库


question 用户问题
  ↓
Embedding 模型 text-embedding-v4
  ↓
query vector 问题向量
  ↓
FAISS Retrieval 相似度检索
  ↓
Return top-k 相关文档块
  ↓
format_docs 拼成 context
  ↓
Prompt 模板填充 {context} 和 {question}
  ↓
qwen3-max
  ↓
RAG answer 最终答案


In [1]:
# check the env
import sys
import os

print("Python executable:", sys.executable)

Python executable: /root/miniconda3/envs/rag_qwen3max/bin/python


In [2]:
# =============================================================================
# 0. Global configuration: imports, credentials, model names, and constants
# =============================================================================

from __future__ import annotations

import os
from typing import Iterable, List, Tuple

import bs4
import numpy as np
import tiktoken

from langchain import hub
from langchain.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------
# API credentials and tracing
# Qwen API: https://bailian.console.aliyun.com/
# LangSmith API: https://docs.langchain.com/langsmith/home
# -----------------------------
os.environ["DASHSCOPE_API_KEY"] = "" # enter your key between " and "
os.environ["LANGCHAIN_API_KEY"] = "" # enter your key

# LangSmith / LangChain tracing compatibility names.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "rag-from-scratch-qwen3-max-jupyter"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]
os.environ["LANGSMITH_PROJECT"] = os.environ["LANGCHAIN_PROJECT"]

# -----------------------------
# DashScope OpenAI-compatible settings
# -----------------------------
DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1" 
GENERATION_MODEL = "qwen3-max" # model ID
EMBEDDING_MODEL = "text-embedding-v4"
EMBEDDING_DIM = 1024 # 每个文本块被 embedding 模型编码成多长的向量

# -----------------------------
# RAG source and demo question
# The "QUESTION" will be used in Retrieval module、generation module and RAG module
# A open blog from the internet: https://lilianweng.github.io/posts/2023-06-23-agent/
# -----------------------------
BLOG_URL = "https://lilianweng.github.io/posts/2023-06-23-agent/"
QUESTION = "What is Task Decomposition? and what is API-Bank?"

print("DASHSCOPE_BASE_URL:", DASHSCOPE_BASE_URL)
print("GENERATION_MODEL:", GENERATION_MODEL)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("EMBEDDING_DIM:", EMBEDDING_DIM)
print("LANGCHAIN_PROJECT:", os.environ["LANGCHAIN_PROJECT"])


USER_AGENT environment variable not set, consider setting it to identify your requests.


DASHSCOPE_BASE_URL: https://dashscope.aliyuncs.com/compatible-mode/v1
GENERATION_MODEL: qwen3-max
EMBEDDING_MODEL: text-embedding-v4
EMBEDDING_DIM: 1024
LANGCHAIN_PROJECT: rag-from-scratch-qwen3-max-jupyter


## 1. Overview: token counting, embedding, and cosine similarity

This module corresponds to the basic preparation in RAG tutorials:

- Use `tiktoken` to count tokens.
- Use DashScope `text-embedding-v4` through an OpenAI-compatible interface.
- Compare a question vector and a document vector using cosine similarity.


In [3]:
# =============================================================================
# 1. Overview module
# =============================================================================

def num_tokens_from_string(text: str, encoding_name: str = "cl100k_base") -> int:
    """Return the number of tokens in a text string using tiktoken."""
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(text))


def cosine_similarity(vec1: Iterable[float], vec2: Iterable[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a = np.asarray(list(vec1), dtype=np.float32)
    b = np.asarray(list(vec2), dtype=np.float32)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def build_embeddings() -> OpenAIEmbeddings:
    """Build DashScope OpenAI-compatible embedding model."""
    return OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url=DASHSCOPE_BASE_URL,
        dimensions=EMBEDDING_DIM,
        chunk_size=10,
        check_embedding_ctx_length=False,
    )


def build_llm() -> ChatOpenAI:
    """Build DashScope OpenAI-compatible chat model."""
    return ChatOpenAI(
        model=GENERATION_MODEL,
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url=DASHSCOPE_BASE_URL,
        temperature=0.1, # 控制生成模型的随机性/发散程度
        max_tokens=2048, # 控制模型最多生成多少输出token
        timeout=120, # API请求最长等待时间 (s)
        max_retries=2, # 控制API请求失败后的最大重试次数
    )


embeddings = build_embeddings()
llm = build_llm()

question_demo = "What kinds of pets do I like?"
document_demo = "My favorite pet is a cat."

print("question:", question_demo)
print("question token count:", num_tokens_from_string(question_demo, "cl100k_base"))

query_result = embeddings.embed_query(question_demo)
document_result = embeddings.embed_query(document_demo)

print("embedding dim:", len(query_result))
print("cosine similarity:", cosine_similarity(query_result, document_result))

chinese_text = "这是一个用于编码解码的示例文本。"
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(chinese_text)
decoded_text = encoding.decode(tokens)

print("Chinese demo tokens:", tokens)
print("Chinese demo decoded:", decoded_text)


question: What kinds of pets do I like?
question token count: 8
embedding dim: 1024
cosine similarity: 0.5299914479255676
Chinese demo tokens: [44388, 21043, 48044, 20379, 27452, 17161, 22656, 1811]
Chinese demo decoded: 这是一个示例文本。


## 2. Indexing: load, split, and store documents

This module implements the indexing stage:

1. Load the Lilian Weng agent blog with `WebBaseLoader`.
2. Parse only the blog title, header, and content with `BeautifulSoup`.
3. Split the document into chunks with `RecursiveCharacterTextSplitter.from_tiktoken_encoder`.
4. Store the chunk embeddings in a FAISS vector store.


In [4]:
# =============================================================================
# 2. Indexing module
# =============================================================================

loader = WebBaseLoader(
    web_paths=(BLOG_URL,),
    bs_kwargs={
        "parse_only": bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    },
)

blog_docs = loader.load()

print(f"loaded docs: {len(blog_docs)}")
if blog_docs:
    print("first doc source:", blog_docs[0].metadata.get("source"))
    print("first doc chars:", len(blog_docs[0].page_content))

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, # 控制每个文本块的最大长度
    chunk_overlap=50, # 控制相邻文本块之间的重叠长度
)

splits = text_splitter.split_documents(blog_docs)

print("num splits:", len(splits))
if splits:
    print("first split preview:", splits[0].page_content[:300].replace("\n", " "))

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
)

print("FAISS vectorstore built.")


loaded docs: 1
first doc source: https://lilianweng.github.io/posts/2023-06-23-agent/
first doc chars: 43047
num splits: 19
first split preview: LLM Powered Autonomous Agents      Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng   Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring
FAISS vectorstore built.


## 3. Retrieval: search relevant chunks

This module implements the retrieval stage:

- Convert the FAISS vector store into a retriever.
- Retrieve the top-k chunks relevant to `QUESTION`.
- Print retrieved chunks to inspect whether the retrieval stage is reasonable.


In [5]:
# =============================================================================
# 3. Retrieval module
# =============================================================================

retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # 每个问题只取最相关的 k 个 chunk
retrieved_docs = retriever.invoke(QUESTION)

print("query:", QUESTION)
print("retrieved docs:")

for idx, doc in enumerate(retrieved_docs, start=1):
    source = doc.metadata.get("source", "unknown")
    print(f"\n--- doc {idx} | source={source} ---")
    print(doc.page_content[:800].strip())


query: What is Task Decomposition? and what is API-Bank?
retrieved docs:

--- doc 1 | source=https://lilianweng.github.io/posts/2023-06-23-agent/ ---
(2) Model selection: LLM distributes the tasks to expert models, where the request is framed as a multiple-choice question. LLM is presented with a list of models to choose from. Due to the limited context length, task type based filtration is needed.
Instruction:

Given the user request and the call command, the AI assistant helps the user to select a suitable model from a list of models to process the user request. The AI assistant merely outputs the model id of the most appropriate model. The output must be in a strict JSON format: "id": "id", "reason": "your detail reason for the choice". We have a list of models for you to choose from {{ Candidate Models }}. Please select one model from the list.

(3) Task execution: Expert models execute on the specific tasks and log results.
Instruc

--- doc 2 | source=https://lilianweng.github.io/

## 4. Generation: prompt + qwen3-max answer

This module implements generation with explicit retrieved context:

- Serialize retrieved documents into a context string.
- Try to load the `rlm/rag-prompt` prompt from LangChain Hub.
- If Hub loading fails, use a local RAG prompt.
- Feed the retrieved context and question into `qwen3-max`.


In [6]:
# =============================================================================
# 4. Generation module
# =============================================================================


def format_docs(docs: List[Document]) -> str:
    """Serialize retrieved documents into a compact context string."""
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\nContent: {doc.page_content}"
        for doc in docs
    )


try:
    prompt = hub.pull("rlm/rag-prompt")
    print("[info] Loaded prompt from LangChain Hub: rlm/rag-prompt")
except Exception as exc:
    print(f"[warn] Take it easy~ hub.pull failed, using local prompt. reason={exc!r}")
    local_template = """Firstly introduce yourself, what's your foundation model? and then answer the question based only on the following context.
If the context does not contain the answer, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""
    prompt = ChatPromptTemplate.from_template(local_template)


context = format_docs(retrieved_docs)

generation_chain = prompt | llm | StrOutputParser()
generated_answer = generation_chain.invoke(
    {
        "context": context,
        "question": QUESTION,
    }
)

print("got retrieved_context:")
print(context)

print("generated answer:")
print(generated_answer)


[warn] Take it easy~ hub.pull failed, using local prompt. reason=ValueError('Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.')
got context:
Source: https://lilianweng.github.io/posts/2023-06-23-agent/
Content: (2) Model selection: LLM distributes the tasks to expert models, where the request is framed as a multiple-choice question. LLM is presented with a list of models to choose from. Due to the limited context length, task type based filtration is needed.
Instruction:

Given the user request and the call command, the AI assistant helps the user to select a suitable model from a list of models to process the user request. The AI assistant merely outputs the model id of the most appropriate model. The output must be in a strict JSON format: "id": "id", "reason": "your detail reason for the choice". We have a list o

## 5. Full RAG chain: Retriever → Prompt → qwen3-max → Parser

This module connects all parts into one LCEL chain:

1. User question enters the chain.
2. Retriever gets relevant chunks.
3. Retrieved chunks are formatted as context.
4. Prompt combines context and question.
5. `qwen3-max` generates the final answer.
6. `StrOutputParser` converts the model response to a plain string.

输入 query = "What is Task Decomposition?"
  ↓
分成两路
  ├── 路径 1：query → retriever → retrieved_docs → format_docs → context
  └── 路径 2：query → RunnablePassthrough → question
  ↓
得到：
{
  "context": 检索到的网页上下文,
  "question": 原始问题
}
  ↓
prompt 格式化
  ↓
qwen3-max 生成
  ↓
StrOutputParser 转成字符串
  ↓
final_answer


In [7]:
# =============================================================================
# 5. Full RAG module
# =============================================================================

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

final_answer = rag_chain.invoke(QUESTION)

print("RAG final answer:")
print(final_answer)


RAG final answer:
I am an AI assistant based on the Qwen model developed by Alibaba Cloud. My purpose is to assist users by providing accurate and contextually relevant information.

Based solely on the provided context:

**Task Decomposition** is a technique used by LLM-powered autonomous agents to break down a complicated task into smaller, manageable steps. It can be achieved through methods like Chain of Thought (CoT), which instructs the model to “think step by step,” or Tree of Thoughts (ToT), which explores multiple reasoning paths at each step. Task decomposition can also be performed using task-specific instructions or human input, and in some approaches like LLM+P, it is outsourced to an external classical planner using PDDL (Planning Domain Definition Language).

**API-Bank** is a benchmark for evaluating the performance of tool-augmented large language models (LLMs). It includes 53 commonly used API tools, a complete tool-augmented LLM workflow, and 264 annotated dialogues 

## References

- LangChain RAG from Scratch 1–4 notebook: https://github.com/langchain-ai/rag-from-scratch/blob/main/rag_from_scratch_1_to_4.ipynb
- LangChain RAG documentation: https://docs.langchain.com/oss/python/langchain/rag
- OpenAI Cookbook tiktoken token counting example: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
